In [1]:
import requests
import urllib3
import re
from bs4 import BeautifulSoup
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

In [2]:
base_url = "https://gurukula.com/sa/mahabharata/the-writing-2/story"

In [3]:
base_response = requests.get(base_url,timeout=10)
raw_html = base_response.text
print(raw_html)

<!DOCTYPE html><html><head><script>!function(f,b,e,v,n,t,s)
{if(f.fbq)return;n=f.fbq=function(){n.callMethod?n.callMethod.apply(n,arguments):n.queue.push(arguments)};if(!f._fbq)f._fbq=n;n.push=n;n.loaded=!0;n.version='2.0';n.queue=[];t=b.createElement(e);t.async=!0;t.src=v;s=b.getElementsByTagName(e)[0];s.parentNode.insertBefore(t,s)}(window,document,'script','https://connect.facebook.net/en_US/fbevents.js');fbq('init','2195783624203625');</script><script src="/rcb9m59p/init.js"></script><meta charSet="utf-8"/><meta name="viewport" content="width=device-width"/><meta http-equiv="Cache-Control" content="max-age=28800"/><script async="" src="https://www.googletagmanager.com/gtag/js?id=G-Q48HNQVBP2"></script><script id="google-analytics" strategy="afterInteractive">
            window.dataLayer = window.dataLayer || [];
            function gtag(){dataLayer.push(arguments);}
            gtag(&#x27;js&#x27;, new Date());
            gtag(&#x27;config&#x27;, &#x27;G-Q48HNQVBP2&#x27;);
     

Parse HTML

In [4]:
print(type(raw_html))

<class 'str'>


In [5]:
from bs4 import BeautifulSoup
import json

soup = BeautifulSoup(raw_html, "html.parser")

next_script = soup.find("script", id="__NEXT_DATA__")

if next_script:
    raw_json = next_script.string
    next_data = json.loads(raw_json)

    print(type(next_data))
else:
    print("No __NEXT_DATA__ found")

<class 'dict'>


In [6]:
page_data = next_data["props"]["pageProps"]["data"]

print(page_data.keys())

dict_keys(['chapterDetailObj', 'epicChapterTitleList', 'currChapterIndexInArray', 'storyBufferData'])


In [7]:
# buffer_data = page_data["storyBufferData"]["Story"]["data"]

# story_html = bytes(buffer_data).decode("utf-8")

# print(story_html)
buffer_data = page_data["storyBufferData"]["Story"]["data"]

raw_bytes = bytes(buffer_data)

for encoding in ["utf-8", "cp1252", "latin-1"]:
    try:
        story_html = raw_bytes.decode(encoding)
        print("Success:", encoding)
        break
    except UnicodeDecodeError:
        print("Failed:", encoding)

Success: utf-8


In [8]:
from bs4 import BeautifulSoup

soup = BeautifulSoup(story_html, "html.parser")

paragraphs = soup.find_all("p")

sanskrit_text = []

for p in paragraphs:
    text = p.get_text(" ", strip=True)

    if text:
        sanskrit_text.append(text)

story = "\n".join(sanskrit_text)

print(story)

भगवान् व्यासः महाभारतं रचितवान्। सः महाभारतकथाम्
अन्येन लेखयितुम् इष्टवान् किन्तु लेखने समर्थः कः इति न ज्ञातवान्। अतः सः ब्रह्मदेवस्य
समीपं गत्वा पृष्टवान्। ब्रह्मणः उपदेशानुसारं गणेशः एव अस्मिन् कार्ये समर्थः इति
ज्ञात्वा गणेशं प्रार्थितवान् व्यासमहर्षिः।
गणेशः अपि प्रार्थितवान् “अहम् अवश्यं
करिष्यामि। किन्तु एकः नियमः अस्ति। अहं लेखनं मध्ये न स्थगयामि। अतः भवान् अपि निरन्तरं
कथयतु” इति ।
व्यासः अपि तत्प्रार्थनाम् अङ्गीकृतवान्।
“हे देव! अहं निरन्तरं कथयामि। किन्तु लेखनसमये प्रतिश्लोकम् अर्थं सम्यक् ज्ञात्वा
एव लेखनीयम्” इति प्रतिप्रार्थनां कृतवान् भगवान् व्यासः।
यदा व्यासः कठिनान् श्लोकान् उक्तवान्
तदा गणेशः श्लोकस्य अर्थावगमनार्थं किञ्चित् स्थगितवान्। तमेव अवकाशं प्रयुज्य व्यासमहर्षिः
अग्रिम-श्लोकान् रचितवान्। एवं प्रकारेण भगवान् व्यासः सम्पूर्णं महाभारतं गणेशाय उक्तवान्।
महाभारते लक्षाधिकाः श्लोकाः सन्ति। विंशति-लक्षाधिकानि
पदानि सन्ति।
अधुना वयं बहूनि काल्पनिक-कथापुस्तकानि
पठामः। तेषु सामान्यतः चत्वारिंशत्-सहस्र-पर्यन्तं पदानि भवन्ति। “Harry Potter” सम्बद्धानि
पुस्तकानि सप्त सन्ति

In [9]:
import re

devanagari_block = re.compile(r'[\u0900-\u097F\u200C\u200D\s।,?"“”]+')

paragraphs = []

for p in soup.find_all("p"):
    text = p.get_text(" ", strip=True)

    if text.startswith("Translated by"):
        continue

    # Extract only Devanagari characters
    matches = devanagari_block.findall(text)

    sanskrit_text = " ".join(matches).strip()

    # remove empty results
    if sanskrit_text:
        paragraphs.append(sanskrit_text)


story = "\n\n".join(paragraphs)

print(story)

भगवान् व्यासः महाभारतं रचितवान्। सः महाभारतकथाम्
अन्येन लेखयितुम् इष्टवान् किन्तु लेखने समर्थः कः इति न ज्ञातवान्। अतः सः ब्रह्मदेवस्य
समीपं गत्वा पृष्टवान्। ब्रह्मणः उपदेशानुसारं गणेशः एव अस्मिन् कार्ये समर्थः इति
ज्ञात्वा गणेशं प्रार्थितवान् व्यासमहर्षिः।

गणेशः अपि प्रार्थितवान् “अहम् अवश्यं
करिष्यामि। किन्तु एकः नियमः अस्ति। अहं लेखनं मध्ये न स्थगयामि। अतः भवान् अपि निरन्तरं
कथयतु” इति ।

व्यासः अपि तत्प्रार्थनाम् अङ्गीकृतवान्।
“हे देव  अहं निरन्तरं कथयामि। किन्तु लेखनसमये प्रतिश्लोकम् अर्थं सम्यक् ज्ञात्वा
एव लेखनीयम्” इति प्रतिप्रार्थनां कृतवान् भगवान् व्यासः।

यदा व्यासः कठिनान् श्लोकान् उक्तवान्
तदा गणेशः श्लोकस्य अर्थावगमनार्थं किञ्चित् स्थगितवान्। तमेव अवकाशं प्रयुज्य व्यासमहर्षिः
अग्रिम श्लोकान् रचितवान्। एवं प्रकारेण भगवान् व्यासः सम्पूर्णं महाभारतं गणेशाय उक्तवान्।

महाभारते लक्षाधिकाः श्लोकाः सन्ति। विंशति लक्षाधिकानि
पदानि सन्ति।

अधुना वयं बहूनि काल्पनिक कथापुस्तकानि
पठामः। तेषु सामान्यतः चत्वारिंशत् सहस्र पर्यन्तं पदानि भवन्ति। “   ” सम्बद्धानि
पुस्तकानि सप्त सन्ति। ते

In [10]:
sanskrit_passage =[]
import re

# Normalize whitespace
story = re.sub(r'\s+', ' ', story).strip()

# Split into Sanskrit sentences
sentences = [
    s.strip() + "।"
    for s in story.split("।")
    if s.strip()
]

for i, sentence in enumerate(sentences, 1):
    sanskrit_passage.append(sentence)

print(sanskrit_passage)

['भगवान् व्यासः महाभारतं रचितवान्।', 'सः महाभारतकथाम् अन्येन लेखयितुम् इष्टवान् किन्तु लेखने समर्थः कः इति न ज्ञातवान्।', 'अतः सः ब्रह्मदेवस्य समीपं गत्वा पृष्टवान्।', 'ब्रह्मणः उपदेशानुसारं गणेशः एव अस्मिन् कार्ये समर्थः इति ज्ञात्वा गणेशं प्रार्थितवान् व्यासमहर्षिः।', 'गणेशः अपि प्रार्थितवान् “अहम् अवश्यं करिष्यामि।', 'किन्तु एकः नियमः अस्ति।', 'अहं लेखनं मध्ये न स्थगयामि।', 'अतः भवान् अपि निरन्तरं कथयतु” इति।', 'व्यासः अपि तत्प्रार्थनाम् अङ्गीकृतवान्।', '“हे देव अहं निरन्तरं कथयामि।', 'किन्तु लेखनसमये प्रतिश्लोकम् अर्थं सम्यक् ज्ञात्वा एव लेखनीयम्” इति प्रतिप्रार्थनां कृतवान् भगवान् व्यासः।', 'यदा व्यासः कठिनान् श्लोकान् उक्तवान् तदा गणेशः श्लोकस्य अर्थावगमनार्थं किञ्चित् स्थगितवान्।', 'तमेव अवकाशं प्रयुज्य व्यासमहर्षिः अग्रिम श्लोकान् रचितवान्।', 'एवं प्रकारेण भगवान् व्यासः सम्पूर्णं महाभारतं गणेशाय उक्तवान्।', 'महाभारते लक्षाधिकाः श्लोकाः सन्ति।', 'विंशति लक्षाधिकानि पदानि सन्ति।', 'अधुना वयं बहूनि काल्पनिक कथापुस्तकानि पठामः।', 'तेषु सामान्यतः चत्वारिंशत् सहस्र पर्यन्तं पदानि भवन

convert to english

In [11]:
import uuid
trans_base_url = "https://dharmamitra.org/bff/api/translation"

In [12]:
import uuid

def _create_payload(sentences):

    return {
        "do_grammar": False,
        "input_encoding": "auto",
        "input_sentence": ", ".join(f"'{s}'" for s in sentences),
        "messages": [
            {
                "parts": [
                    {
                        "type": "text",
                        "text": ", ".join(f"'{s}'" for s in sentences)
                    }
                ],
                "id": str(uuid.uuid4()),
                "role": "user"
            }
        ],
        "mode": "normal",
        "target_lang": "english"
    }

In [13]:
    def _create_headers():
        return{
                "Accept": "text/event-stream",
                "Content-Type": "application/json",
                "Origin": "https://dharmamitra.org",
                "Referer": "https://dharmamitra.org/"
        }

In [14]:
trans_req = requests.post(trans_base_url,json=_create_payload(sanskrit_passage),headers=_create_headers(),stream =True)
print(trans_req.text)

data: {"type":"start","messageId":"uIDtMtgdIteoEXZH"}

data: {"type":"start-step"}

data: {"type":"text-start","id":"txt-0"}

data: {"type":"text-delta","id":"txt-0","delta":"The venerable"}

data: {"type":"text-delta","id":"txt-0","delta":" VyÄsa composed the MahÄbhÄrata.\n\nHe desired to have the story of the"}

data: {"type":"text-delta","id":"txt-0","delta":" MahÄbhÄrata written down by another, but he did not know who would be capable of such writing.\n\nTherefore, he"}

data: {"type":"text-delta","id":"txt-0","delta":" went near Lord BrahmÄ and questioned him.\n\nFollowing the advice of BrahmÄ, the great sage VyÄsa,"}

data: {"type":"text-delta","id":"txt-0","delta":" realizing that Gaá¹eÅa alone was capable of this task, offered a prayer to him.\n\nGaá¹eÅa also requested"}

data: {"type":"text-delta","id":"txt-0","delta":", \"I shall certainly do it.\n\nHowever, there is one rule.\n\nI will not stop the writing in"}

data: {"type":"text-delta","id":"txt-0","delta":" 

In [15]:
 def _extract_data(response):
        full_text = ""

        for line in response.iter_lines():
            if not line:
                continue

            line = line.decode("utf-8")

            if line.startswith("data: "):
                event_data = line[6:]

                if event_data == "[DONE]":
                    break

                try:
                    obj = json.loads(event_data)
                except json.JSONDecodeError:
                    continue

                if obj.get("type") == "text-delta":
                    full_text += obj.get("delta", "")

        return full_text

In [16]:
translated_raw_data = _extract_data(trans_req)

translated_english_passage = " ".join(translated_raw_data.split())

print(translated_english_passage)

The venerable Vyāsa composed the Mahābhārata. He desired to have the story of the Mahābhārata written down by another, but he did not know who would be capable of such writing. Therefore, he went near Lord Brahmā and questioned him. Following the advice of Brahmā, the great sage Vyāsa, realizing that Gaṇeśa alone was capable of this task, offered a prayer to him. Gaṇeśa also requested, "I shall certainly do it. However, there is one rule. I will not stop the writing in the middle. Therefore, you must also speak continuously." Vyāsa also accepted that request. "O Lord, I shall speak continuously. But at the time of writing, you must write only after correctly understanding the meaning of every verse," the venerable Vyāsa made this counter-request. Whenever Vyāsa spoke difficult verses, Gaṇeśa paused for a moment to grasp the meaning of the verse. Utilizing that very opportunity, the great sage Vyāsa composed the subsequent verses. In this manner, the venerable Vyāsa narrated the entire 

In [17]:
import unicodedata

def normalize_diacritics(text):
    return "".join(
        c for c in unicodedata.normalize("NFD", text)
        if unicodedata.category(c) != "Mn"
    )

In [18]:
english_passage = normalize_diacritics(translated_english_passage)

print(english_passage)

The venerable Vyasa composed the Mahabharata. He desired to have the story of the Mahabharata written down by another, but he did not know who would be capable of such writing. Therefore, he went near Lord Brahma and questioned him. Following the advice of Brahma, the great sage Vyasa, realizing that Ganesa alone was capable of this task, offered a prayer to him. Ganesa also requested, "I shall certainly do it. However, there is one rule. I will not stop the writing in the middle. Therefore, you must also speak continuously." Vyasa also accepted that request. "O Lord, I shall speak continuously. But at the time of writing, you must write only after correctly understanding the meaning of every verse," the venerable Vyasa made this counter-request. Whenever Vyasa spoke difficult verses, Ganesa paused for a moment to grasp the meaning of the verse. Utilizing that very opportunity, the great sage Vyasa composed the subsequent verses. In this manner, the venerable Vyasa narrated the entire 

In [19]:
import re

def clean_translation(text):
    # Normalize whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    # Remove empty quoted fragments
    text = re.sub(r'["“”\']\s*["“”\']', '', text)

    # Remove quotes containing only punctuation
    text = re.sub(r'["“”\']\s*[,.;:!?।]+\s*["“”\']', '', text)

    # Remove duplicated punctuation
    text = re.sub(r'([,.!?])\s*\1+', r'\1', text)

    # Fix punctuation spacing
    text = re.sub(r'\s+([,.!?])', r'\1', text)

    # Remove orphan punctuation
    text = re.sub(r'\s+[,.;:]+\s*(?=[A-Za-z])', ' ', text)

    # Clean quotation spacing
    text = re.sub(r'\s+([\'"])', r'\1', text)
    text = re.sub(r'([\'"])\s+', r'\1 ', text)

    return text.strip()

In [20]:
cleaned_english_passage = clean_translation(english_passage)
print(cleaned_english_passage)

The venerable Vyasa composed the Mahabharata. He desired to have the story of the Mahabharata written down by another, but he did not know who would be capable of such writing. Therefore, he went near Lord Brahma and questioned him. Following the advice of Brahma, the great sage Vyasa, realizing that Ganesa alone was capable of this task, offered a prayer to him. Ganesa also requested,"I shall certainly do it. However, there is one rule. I will not stop the writing in the middle. Therefore, you must also speak continuously." Vyasa also accepted that request."O Lord, I shall speak continuously. But at the time of writing, you must write only after correctly understanding the meaning of every verse," the venerable Vyasa made this counter-request. Whenever Vyasa spoke difficult verses, Ganesa paused for a moment to grasp the meaning of the verse. Utilizing that very opportunity, the great sage Vyasa composed the subsequent verses. In this manner, the venerable Vyasa narrated the entire Ma